# 6.4 Grad-CAM 可视化

本 notebook 介绍 Grad-CAM（Gradient-weighted Class Activation Mapping）算法，用于解释 CNN 的决策依据。

**学习目标：**
- 了解 CAM 算法家族的演进
- 理解 Grad-CAM 的核心原理
- 掌握使用 forward hook + backward hook 的实现方式
- 完成一个完整的 GradCAM 类
- 理解 Grad-CAM 在模型调试中的实际价值

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch 版本: {torch.__version__}")

## 1. CAM 算法演进

类激活图（Class Activation Mapping）是一类用于解释 CNN 决策的可视化方法：

| 算法 | 年份 | 核心思想 | 限制 |
|------|------|----------|------|
| **CAM** | 2016 | 利用 GAP 层的权重加权最后的特征图 | 要求模型必须有 GAP 层 |
| **Grad-CAM** | 2017 | 用梯度代替 GAP 权重，适用于任意 CNN | 对多目标场景效果有限 |
| **Grad-CAM++** | 2018 | 使用高阶梯度加权，改进多目标场景 | 计算成本更高 |

### CAM 的局限
CAM 要求模型最后一层必须是 GAP（Global Average Pooling）+ FC，这限制了它的适用范围。

### Grad-CAM 的改进
Grad-CAM 的核心发现：**梯度可以代替 GAP 的权重**，因此适用于任意 CNN 结构。

## 2. Grad-CAM 原理

Grad-CAM 通过以下步骤生成类激活图：

1. **前向传播**: 获取目标卷积层的特征图 $A^k$（$k$ 为通道索引）
2. **反向传播**: 计算目标类别分数 $y^c$ 对特征图的梯度 $\frac{\partial y^c}{\partial A^k}$
3. **计算权重**: 对梯度做全局平均池化，得到每个通道的重要性权重
   $$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$
4. **加权求和**: 用权重对特征图加权求和
   $$L^c_{Grad-CAM} = ReLU\left(\sum_k \alpha_k^c A^k\right)$$
5. **ReLU**: 只保留正贡献区域
6. **归一化**: 将结果归一化到 [0, 1] 并上采样到输入尺寸

## 3. 定义示例模型

In [ ]:
class SimpleCNN(nn.Module):
    """用于 Grad-CAM 演示的简单 CNN。"""
    
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            # 第一层: 3 -> 16
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
            
            # 第二层: 16 -> 32
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
            
            # 第三层: 32 -> 64（目标层）
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=5)
print("模型结构:")
print(model)
print(f"\n目标层: features 中的最后一个卷积层")

## 4. 逐步实现 Grad-CAM

分步骤演示 Grad-CAM 的完整流程。

In [ ]:
# 步骤1: 注册 hook 捕获特征图和梯度
feature_maps = {}  # 存储特征图
gradients = {}     # 存储梯度

def forward_hook(module, input, output):
    """前向 hook: 捕获特征图。"""
    feature_maps["target"] = output.detach()

def backward_hook(module, grad_input, grad_output):
    """反向 hook: 捕获梯度。"""
    gradients["target"] = grad_output[0].detach()

# 目标层: features 中最后一个 ReLU 之前的 Conv2d
# 即 features[8] (第三个 Conv2d)
target_layer = model.features[10]  # 最后一个 ReLU
print(f"目标层: {target_layer}")

# 注册 hook
fwd_hook = target_layer.register_forward_hook(forward_hook)
bwd_hook = target_layer.register_full_backward_hook(backward_hook)
print("Hook 已注册")

In [ ]:
# 步骤2: 前向传播
test_input = torch.randn(1, 3, 32, 32)
model.eval()
output = model(test_input)

print(f"输入形状: {test_input.shape}")
print(f"输出形状: {output.shape}")
print(f"输出值: {output.data}")
print(f"预测类别: {output.argmax(dim=1).item()}")
print(f"\n捕获的特征图形状: {feature_maps['target'].shape}")

In [ ]:
# 步骤3: 反向传播（针对目标类别）
target_class = output.argmax(dim=1).item()  # 使用预测类别
print(f"目标类别: {target_class}")

# 创建 one-hot 向量
one_hot = torch.zeros_like(output)
one_hot[0, target_class] = 1.0

# 反向传播
model.zero_grad()
output.backward(gradient=one_hot)

print(f"捕获的梯度形状: {gradients['target'].shape}")

In [ ]:
# 步骤4: 计算通道权重（梯度全局平均池化）
grad = gradients["target"]  # (1, C, H, W)
weights = grad.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1) 全局平均池化

print(f"梯度形状: {grad.shape}")
print(f"权重形状: {weights.shape}")
print(f"权重值 (前 8 个通道): {weights[0, :8, 0, 0].tolist()}")

In [ ]:
# 步骤5: 加权求和 + ReLU
feat = feature_maps["target"]  # (1, C, H, W)

# 加权求和
cam = (weights * feat).sum(dim=1, keepdim=True)  # (1, 1, H, W)

# ReLU: 只保留正贡献
cam = F.relu(cam)

print(f"加权求和后形状: {cam.shape}")
print(f"值范围: [{cam.min():.4f}, {cam.max():.4f}]")

In [ ]:
# 步骤6: 归一化 + 上采样
# 归一化到 [0, 1]
cam_min = cam.min()
cam_max = cam.max()
if cam_max - cam_min > 0:
    cam = (cam - cam_min) / (cam_max - cam_min)

# 上采样到输入尺寸
cam_resized = F.interpolate(
    cam, size=(32, 32), mode="bilinear", align_corners=False
)

print(f"归一化后: [{cam.min():.4f}, {cam.max():.4f}]")
print(f"上采样后形状: {cam_resized.shape}")

# 可视化
cam_np = cam_resized[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# 原始输入
input_display = test_input[0].permute(1, 2, 0).numpy()
input_display = (input_display - input_display.min()) / (input_display.max() - input_display.min())
axes[0].imshow(input_display)
axes[0].set_title("Input Image")
axes[0].axis("off")

# Grad-CAM 热力图
axes[1].imshow(cam_np, cmap="jet")
axes[1].set_title(f"Grad-CAM (class={target_class})")
axes[1].axis("off")

# 叠加
axes[2].imshow(input_display)
axes[2].imshow(cam_np, cmap="jet", alpha=0.5)
axes[2].set_title("Overlay")
axes[2].axis("off")

plt.suptitle("Grad-CAM Step-by-Step Result", fontsize=14)
plt.tight_layout()
plt.show()

# 清理 hook
fwd_hook.remove()
bwd_hook.remove()

## 5. 完整 GradCAM 类

将上述步骤封装成可复用的类。

In [ ]:
class GradCAM:
    """Grad-CAM 可视化工具。
    
    使用梯度加权的类激活图来可视化 CNN 的决策依据。
    """
    
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        """初始化 GradCAM。
        
        Args:
            model: CNN 模型
            target_layer: 要可视化的目标卷积层
        """
        self.model = model
        self.target_layer = target_layer
        self.feature_maps = None
        self.gradients = None
        
        # 注册 hook
        self._fwd_hook = target_layer.register_forward_hook(self._forward_hook)
        self._bwd_hook = target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        """捕获前向传播的特征图。"""
        self.feature_maps = output.detach()
    
    def _backward_hook(self, module, grad_input, grad_output):
        """捕获反向传播的梯度。"""
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor: torch.Tensor, 
                 target_class: int = None) -> np.ndarray:
        """生成 Grad-CAM 热力图。
        
        Args:
            input_tensor: 输入图像, shape (1, C, H, W)
            target_class: 目标类别索引, None 表示使用预测类别
        
        Returns:
            归一化的 CAM 热力图, shape (H, W), 值域 [0, 1]
        """
        # 前向传播
        self.model.eval()
        output = self.model(input_tensor)
        
        # 确定目标类别
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # 反向传播
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot)
        
        # 计算通道权重: 对梯度做全局平均池化
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        
        # 加权求和 + ReLU
        cam = (weights * self.feature_maps).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = F.relu(cam)
        
        # 上采样到输入尺寸
        h, w = input_tensor.shape[2:]
        cam = F.interpolate(cam, size=(h, w), mode="bilinear", align_corners=False)
        
        # 归一化到 [0, 1]
        cam = cam[0, 0]  # (H, W)
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 0:
            cam = (cam - cam_min) / (cam_max - cam_min)
        
        return cam.numpy()
    
    def remove_hooks(self):
        """移除所有 hook。"""
        self._fwd_hook.remove()
        self._bwd_hook.remove()

print("GradCAM 类定义完成")

In [ ]:
# 使用 GradCAM 类
model = SimpleCNN(num_classes=5)
target_layer = model.features[10]  # 最后一个 ReLU

gradcam = GradCAM(model, target_layer)

# 生成多个输入的 Grad-CAM
torch.manual_seed(42)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # 随机输入
    test_input = torch.randn(1, 3, 32, 32)
    
    # 生成 CAM
    cam = gradcam.generate(test_input)
    pred_class = model(test_input).argmax(dim=1).item()
    
    # 显示输入
    img = test_input[0].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Input {i+1}")
    axes[0, i].axis("off")
    
    # 显示 CAM
    axes[1, i].imshow(img)
    axes[1, i].imshow(cam, cmap="jet", alpha=0.5)
    axes[1, i].set_title(f"Pred: class {pred_class}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Grad-CAM", fontsize=12)
plt.suptitle("Grad-CAM Visualization (Random Inputs)", fontsize=14)
plt.tight_layout()
plt.show()

gradcam.remove_hooks()

## 6. 对比不同类别的 Grad-CAM

对同一张图片，计算不同类别的 Grad-CAM，观察模型关注区域的差异。

In [ ]:
model = SimpleCNN(num_classes=5)
target_layer = model.features[10]
gradcam = GradCAM(model, target_layer)

# 固定一个输入
torch.manual_seed(100)
test_input = torch.randn(1, 3, 32, 32)

# 对 5 个类别分别生成 Grad-CAM
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

# 原始输入
img = test_input[0].permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min())
axes[0].imshow(img)
axes[0].set_title("Input")
axes[0].axis("off")

# 各类别的 CAM
for cls in range(5):
    cam = gradcam.generate(test_input, target_class=cls)
    axes[cls + 1].imshow(img)
    axes[cls + 1].imshow(cam, cmap="jet", alpha=0.5)
    axes[cls + 1].set_title(f"Class {cls}")
    axes[cls + 1].axis("off")

plt.suptitle("Same Input, Different Target Classes", fontsize=14)
plt.tight_layout()
plt.show()

gradcam.remove_hooks()

print("观察: 不同类别的 Grad-CAM 关注不同的图像区域")
print("这说明模型对不同类别学到了不同的判别特征")

## 7. 实际应用：Shortcut Learning 检测

Grad-CAM 的一个重要应用是检测模型是否学到了**捷径特征（shortcut learning）**。

例如：
- 分类「牛」时，模型实际关注的是草地背景
- 分类「鲨鱼」时，模型关注的是海水颜色
- 医学影像中，模型关注的是标注标记而非病灶

通过 Grad-CAM 可以快速发现这类问题。

In [ ]:
# 模拟一个 shortcut learning 场景
print("Shortcut Learning 检测流程:")
print("=" * 50)
print()
print("1. 训练模型并观察准确率")
print("   - 如果训练集准确率很高，但在新数据上效果差")
print("   - 这可能是 shortcut learning 的信号")
print()
print("2. 使用 Grad-CAM 检查模型关注区域")
print("   - 选择多张正确分类的样本")
print("   - 生成 Grad-CAM 热力图")
print("   - 检查模型是否关注目标物体")
print()
print("3. 常见的 shortcut 模式")
print("   - 关注背景而非前景物体")
print("   - 关注图像边缘或角落的水印/标记")
print("   - 关注特定的颜色/亮度区域")
print()
print("4. 解决方案")
print("   - 数据增强: 随机裁剪、颜色抖动、背景替换")
print("   - 数据清洗: 移除有偏差的样本")
print("   - 正则化: 鼓励模型关注更广泛的区域")
print("   - 对比学习: 学习更鲁棒的特征表示")

## 8. 总结

### Grad-CAM 核心流程
1. 注册 forward hook 捕获特征图
2. 注册 backward hook 捕获梯度
3. 前向传播获取输出
4. 针对目标类别反向传播
5. 梯度全局平均池化得到通道权重
6. 加权求和 + ReLU + 归一化

### 关键代码
```python
# 核心三步
weights = gradients.mean(dim=[2, 3], keepdim=True)  # 全局平均池化
cam = (weights * feature_maps).sum(dim=1)            # 加权求和
cam = F.relu(cam)                                     # 只保留正贡献
```

### 实际价值
- **模型调试**: 检查模型是否关注正确的区域
- **Shortcut 检测**: 发现模型是否学到了虚假相关
- **可解释性**: 向非技术人员解释模型决策
- **模型比较**: 对比不同模型的关注区域差异

---

## 练习题

**练习1（基础）：** 修改 GradCAM 类，使其支持传入层名称（字符串）而非层对象。提示：使用 `dict(model.named_modules())` 获取层。

**练习2（进阶）：** 对 SimpleCNN 的三个不同卷积层分别生成 Grad-CAM，对比浅层和深层的关注区域差异。

**练习3（挑战）：** 实现一个简化版的 Grad-CAM++：使用二阶梯度（梯度的 ReLU）替代简单的全局平均池化来计算权重。对比 Grad-CAM 和 Grad-CAM++ 的结果差异。